# 🔧 Predictive Maintenance — Visualization Notebook

This notebook is the **visualization companion** to the production codebase.
It covers all plots from Milestones 1–3:
- Feature correlation heatmap (M1)
- Degradation time-series leading to failure (M2)
- PCA scatter: Normal vs Failure clusters (M2)
- Confusion matrices for all models (M3)
- Feature importance chart (M3)

> **Note:** For production training and model saving, run `main.py` locally.
> This notebook imports from `src/` — mount or clone your repo first.

In [ ]:
# ── 0. Colab Setup (skip if running locally) ────────────────────────────────
# Uncomment the block below if running on Google Colab

# !git clone https://github.com/YOUR_USERNAME/predictive_maintenance.git
# %cd predictive_maintenance
# !pip install -r requirements.txt -q

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.append('..')  # Ensure src/ is importable when running from notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

from src.utils import load_config, load_artifact
from src.data_preprocessing import run_preprocessing
from src.feature_engineering import run_feature_engineering
from src.model_training import split_data

# Shared style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

config = load_config('../config/config.yaml')
print('Config loaded ✅')

In [ ]:
# ── 2. Load & Prepare Data ────────────────────────────────────────────────────
df_clean = run_preprocessing('../config/config.yaml')
df_eng = run_feature_engineering(df_clean, config)

print(f'Engineered shape: {df_eng.shape}')
print(f'Failure rate: {df_eng["Machine_Failure"].mean()*100:.2f}%')
df_eng.head(3)

## 📊 Milestone 1 — Correlation Heatmap

In [ ]:
# Drop boolean dummy columns from the heatmap for readability
numeric_cols = df_eng.select_dtypes(include='number').columns

plt.figure(figsize=(14, 10))
sns.heatmap(
    df_eng[numeric_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 📈 Milestone 2 — Degradation Pattern Before Failure

In [ ]:
# Plot rolling temperature in the 50 steps before the FIRST failure event
first_failure_idx = df_eng[df_eng['Machine_Failure'] == 1].index[0]
start_idx = max(0, first_failure_idx - 50)
window = df_eng.iloc[start_idx : first_failure_idx + 5]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Panel A: Rolling Average Temperature
axes[0].plot(window.index, window['Rolling_Avg_Temp'], color='steelblue', label='Rolling Avg Temp')
axes[0].axvline(x=first_failure_idx, color='red', linestyle='--', label='⚠ Machine Failed')
axes[0].set_ylabel('Temperature (K)')
axes[0].set_title('Degradation Pattern Leading Up to First Failure', fontweight='bold')
axes[0].legend()

# Panel B: Rolling Std of Torque (instability)
axes[1].plot(window.index, window['Rolling_Std_Torque'], color='darkorange', label='Rolling Std Torque')
axes[1].axvline(x=first_failure_idx, color='red', linestyle='--')
axes[1].set_ylabel('Torque Std Dev')
axes[1].set_xlabel('Time Step')
axes[1].legend()

plt.tight_layout()
plt.show()

## 🔬 Milestone 2 — PCA: Normal vs Failure Clusters

In [ ]:
# PCA is ONLY for this visualization — not used in model training
features = df_eng.drop('Machine_Failure', axis=1)
target = df_eng['Machine_Failure']

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled)
explained = pca.explained_variance_ratio_.sum() * 100

pca_df = pd.DataFrame({
    'PC1': pca_result[:, 0],
    'PC2': pca_result[:, 1],
    'Machine_Failure': target.values
})

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2',
    hue='Machine_Failure',
    palette={0: 'mediumseagreen', 1: 'crimson'},
    alpha=0.5, s=20
)
plt.title(f'PCA — Normal vs Failure Clusters\n(Explained Variance: {explained:.1f}%)', fontweight='bold')
plt.legend(title='Machine Failure', labels=['Normal (0)', 'Failure (1)'])
plt.tight_layout()
plt.show()
print(f'Explained variance by 2 PCs: {explained:.2f}%')

## 🏆 Milestone 3 — Confusion Matrices & Feature Importance

> Run `python main.py` first to generate saved models in `/models`.

In [ ]:
# Load saved models and test set
_, _, X_test, _, _, y_test = split_data(df_eng, config)

artifact_cfg = config['artifacts']
model_dir = '../' + artifact_cfg['model_dir']

models = {
    'Random Forest': load_artifact(f"{model_dir}/{artifact_cfg['rf_model_name']}"),
    'XGBoost Base':  load_artifact(f"{model_dir}/{artifact_cfg['xgb_model_name']}"),
    'XGBoost Optimized': load_artifact(f"{model_dir}/{artifact_cfg['best_model_name']}"),
}
print('Models loaded ✅')

In [ ]:
# Plot all confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cmaps = ['Blues', 'Purples', 'Greens']

for ax, (name, model), cmap in zip(axes, models.items(), cmaps):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance from Random Forest
rf_model = models['Random Forest']
importances = pd.Series(
    rf_model.feature_importances_,
    index=X_test.columns
).sort_values(ascending=True)

plt.figure(figsize=(9, 7))
importances.tail(15).plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 15 Feature Importances — Random Forest', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()